# Qwen3-0.6B × FlashAttention-backed AttentionBackend (educational)

**자매 프로젝트**: `../flashinfer/` — CSR 변환 + plan/run, `../ptr/vllm_unified/` — 직접 작성한 Triton 커널.

이 노트북은 `flashinfer` 노트북과 **같은 조직**을 유지하며, 다른 것은 두 곳뿐:

- **cell 7**: CSR 변환 데모 없음 → **pass-through builder 데모** (zero-cost builder 대조)
- **cell 10**: LLM 로드 시 `MyFlashAttnBackend` 가 CUSTOM 슬롯에 올라감

나머지 — plugin entry point, KV cache shape, observability 로그 관찰 포인트 — 는 동일.

**이 프로젝트의 이야기**: `flash_attn_varlen_func` 는 vLLM 의 `block_table` 과 `cu_seqlens` 를 
그대로 받으므로 빌더가 공짜(pass-through)다 — CSR 변환도, `plan()` 도, workspace 도 없다.
단일 varlen 호출로 prefill + decode + chunked 를 통합 처리하는 점은 `vllm_unified` 와 같다.

## 준비물 & 설치

- CUDA GPU 필수 (Blackwell/Hopper/Ampere). FA2 는 SM 8.0+ 지원.
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)
- `flash-attn` (PyPI) 별도 설치 **불필요** — `vllm.vllm_flash_attn` 이 번들에 포함

```bash
# 노트북 디렉토리에서 (entry point 자동 등록 원할 때만)
pip install -e .
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = flash_attn_attention_backend:register`)
가 vLLM 프로세스에서 자동으로 `register()` 를 호출해 준다. 노트북 안에서 수동 호출 불필요.

> **참고**: flashinfer 노트북과 달리 PATH hack (ninja) 이 불필요하다. FA2 는 JIT 컴파일이
> 없어서 ninja 가 필요하지 않다.

In [1]:
import torch, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

# FA varlen import 확인 (PyPI flash-attn 아닌 vLLM 번들)
from vllm.vllm_flash_attn import flash_attn_varlen_func
print('flash_attn_varlen_func (vllm bundle): OK')

assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)

cuda: True
gpu: NVIDIA GeForce RTX 5090
vllm: 0.19.1
flash_attn_varlen_func (vllm bundle): OK


## Qwen3-0.6B 구조 요약

| 항목 | 값 | FA2 제약 충족 여부 |
|---|---|---|
| hidden_size | 1024 | — |
| Q heads | 16 | — |
| KV heads | 8 (GQA 2:1) | OK |
| head_dim | 128 | OK (FA2 허용: % 8 == 0, <= 256) |
| layers | 28 | — |

vLLM `block_size` 기본값 16 도 FA2 허용 조건 (`% 16 == 0`) 에 부합.

In [2]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

assert hd % 8 == 0 and hd <= 256, 'FA2 는 head_dim % 8 == 0 and head_dim <= 256 필요'

/home/osehn/orchestrate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


hidden: 1024
Q heads: 16 / KV heads: 8
head_dim: 128
layers: 28


## FlashAttn 백엔드의 핵심 — zero-cost builder + single launch

`flash_attn_varlen_func` 의 paged-KV 인터페이스는 vLLM 메타데이터와 1:1 대응:

```
CommonAttentionMetadata 필드      flash_attn_varlen_func 인자
─────────────────────────────────────────────────────────
query_start_loc   (B+1,) int32 → cu_seqlens_q
seq_lens          (B,)   int32 → seqused_k
block_table_tensor (B,M) int32 → block_table
max_query_len     int          → max_seqlen_q
max_seq_len       int          → max_seqlen_k
```

따라서 `build()` 는 필드 이름만 바꾸는 pass-through 다. CSR 변환 없음, `plan()` 없음,
workspace 없음. flashinfer 빌더 (~100줄) 와 비교하면 이 프로젝트의 빌더는 ~15줄.

**vllm_unified 와의 차이**: vllm_unified 도 single-launch 이지만 커널을 직접 작성했다.
이 프로젝트는 외부 라이브러리가 그 역할을 하면서 빌더도 공짜가 된 경우다.

## KV layout 비대칭 — 가르침 포인트

이 백엔드는 `get_kv_cache_shape()` 가 `(2, num_blocks, block_size, Hkv, D)` 를 반환한다:

```python
# 이 백엔드 (flashattn):
key_cache, value_cache = kv_cache.unbind(0)   # dim-0 = K/V

# 형제 flashinfer / vllm_unified:
key_cache, value_cache = kv_cache.unbind(1)   # dim-1 = K/V
```

`unbind` 축이 다르다 — 혼동하면 K/V 가 뒤바뀌어 조용히 틀린 결과가 나온다.
vLLM `AttentionBackend` 는 각 backend 가 자신의 레이아웃을 `get_kv_cache_shape` 로
선언하므로 형제끼리도 레이아웃이 다를 수 있다.

In [3]:
# Pass-through builder 데모 (no-op CSR conversion)
# flashinfer 에서는 이 자리에 _build_paged_csr 데모가 있었다.
# flashattn 은 변환이 없으므로 builder 가 하는 일을 직접 확인한다.

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('flash_attn_attention_backend.py')))

import flash_attn_attention_backend as m

print('MyFlashAttnMetadata fields:')
import dataclasses
for f in dataclasses.fields(m.MyFlashAttnMetadata):
    print(f'  {f.name}: {f.type}')

print()
print('Builder has no workspace, no wrappers:')
print('  _WORKSPACE_BYTES: NOT PRESENT (zero-cost)')
print('  plan() step: NONE')
print()
print('flashinfer builder had: CSR conversion + plan() + 128 MiB workspace')
print('flashattn builder has: field rename only (~15 lines)')

MyFlashAttnMetadata fields:
  num_actual_tokens: int
  max_query_len: int
  max_seq_len: int
  query_start_loc: torch.Tensor
  seq_lens: torch.Tensor
  block_table: torch.Tensor
  slot_mapping: torch.Tensor

Builder has no workspace, no wrappers:
  _WORKSPACE_BYTES: NOT PRESENT (zero-cost)
  plan() step: NONE

flashinfer builder had: CSR conversion + plan() + 128 MiB workspace
flashattn builder has: field rename only (~15 lines)


## Plugin 연결

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_flashattn_backend = "flash_attn_attention_backend:register"
```

**Backend 의 단일 launch 로직** (`MyFlashAttnImpl.forward`):
```python
key_cache, value_cache = kv_cache.unbind(0)   # dim-0 K/V split
triton_reshape_and_cache_flash(key[:N], value[:N], key_cache, value_cache, ...)

flash_attn_varlen_func(
    q=query[:N], k=key_cache, v=value_cache, out=output[:N],
    cu_seqlens_q=attn_metadata.query_start_loc,   # pass-through
    seqused_k=attn_metadata.seq_lens,              # pass-through
    block_table=attn_metadata.block_table,         # pass-through
    ...
)
```

flashinfer 의 `plan/run × 2 wrapper` 대비, 이 backend 는 단 1번의 varlen 호출로 끝.

In [4]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

import flash_attn_attention_backend
flash_attn_attention_backend.register()

# vllm_unified / flashinfer 와 동일: max_num_batched_tokens=64 로 chunked prefill trigger.
# 이 backend 도 single varlen call 이 chunked 를 처리한다.
llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
    max_num_batched_tokens=64,   # chunked prefill trigger
)

INFO 05-07 23:35:56 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'max_num_batched_tokens': 64, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'attention_backend': <AttentionBackendEnum.CUSTOM: None>}
INFO 05-07 23:35:58 [model.py:549] Resolved architecture: Qwen3ForCausalLM
WARNING 05-07 23:35:58 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-07 23:35:58 [model.py:1678] Using max model len 2048
INFO 05-07 23:35:58 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=64.
INFO 05-07 23:35:58 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-07 23:35:58 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-07 23:35:58 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-07 23:35:59 

(EngineCore pid=2630315) Process EngineCore:
(EngineCore pid=2630315) Traceback (most recent call last):
(EngineCore pid=2630315)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=2630315)     self.run()
(EngineCore pid=2630315)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=2630315)     self._target(*self._args, **self._kwargs)
(EngineCore pid=2630315)   File "/home/osehn/orchestrate/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1112, in run_engine_core
(EngineCore pid=2630315)     raise e
(EngineCore pid=2630315)   File "/home/osehn/orchestrate/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1082, in run_engine_core
(EngineCore pid=2630315)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=2630315)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=2630315)   File "/home/osehn/orchestra

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
# vllm_unified / flashinfer 와 동일한 prompt 셋으로 결과 비교 가능.
long_prompt = (
    'In the long history of artificial intelligence research, from the early '
    'symbolic AI of the 1950s through the neural network revival of the 1980s, '
    'the deep learning breakthroughs of the 2010s, and the transformer-based '
    'large language models of the 2020s, one theme has remained constant: '
    'the answer is'
)

prompts = [
    'The capital of France is',
    long_prompt,
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}] (prompt {len(prompts[i])} chars) {o.outputs[0].text[:80]}')

## 검증 — single launch 가 모든 타입을 처리한다

엔진 코어 stderr 에 다음 형식의 로그가 찍혀있어야:

```
MyFlashAttnImpl.forward fired num_seqs=N max_q_len=Q chunked=C tokens=T
```

**flashinfer 와의 관찰 포인트 대조**:

| | flashinfer | flashattn (이 프로젝트) |
|---|---|---|
| 로그 형식 | `prefill=P decode=D` 분리 | `chunked=C` 만 (no split) |
| forward 당 launch 수 | 최대 2 | **항상 1** |
| chunked 처리 | prefill_wrapper 로 합류 | **동일 varlen call** |

`chunked > 0` 인 forward 가 관찰되면 단일 launch 가 chunked prefill 도 처리함을 확인.

In [ ]:
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyFlashAttnBackend' in path
print('OK — plugin 등록 확인')
print()
print('FlashAttn 실행 증거 = 위 cell 출력 바로 위의 `MyFlashAttnImpl.forward fired` 로그')
print('→ chunked=C > 0 인 줄이 있으면 단일 launch 가 chunked prefill 도 처리한 것')

## 위치 & 마무리

```
vllm_attn/
  ├─ ptr/vllm_unified/    Triton 커널 1개, 직접 작성, single launch
  ├─ flashinfer/          FlashInfer 외부 wheel, split-dispatch + CSR + plan/run
  └─ flashattn/  (← 여기)  FA2 vLLM 번들, single launch, zero-cost builder
```

**세 프로젝트를 나란히 diff 해 보면**:

1. **Metadata 필드 수**: flashattn = flashinfer (7 필드) = vllm_unified (7 필드) — 셋 다 동일.
   차이는 Builder 가 그것을 어떻게 만드느냐에 있다.

2. **Builder 비용**: flashattn (pass-through) < vllm_unified (pass-through) ≪ flashinfer (CSR)

3. **Impl.forward launch 수**: flashattn = vllm_unified = 1, flashinfer ≤ 2

**단순화된 것 (이 프로젝트)**:
- `cudagraph_support = NEVER` — CUDA Graph 미지원
- cascade (공유 프리픽스) 미지원
- `fa_version=2` 고정 — FA3 (sm_90+ 전용) 미사용
- fp8 KV cache 미지원